In [0]:
%sql
-- ============================================================
-- PASO 2: Crear el esquema Landing
-- ============================================================
-- Crear catalogo si no existe
--Create catalog bootcamp;
-- Usar tu catálogo
USE CATALOG bootcamp;
-- Crear esquema Landing
CREATE SCHEMA IF NOT EXISTS bootcamp.landing
COMMENT 'Esquema para archivos';
-- Crear esquema Raw
CREATE SCHEMA IF NOT EXISTS bootcamp.bronze
COMMENT 'Esquema para datos crudos sin procesar';
-- Verificar
SHOW SCHEMAS;

In [0]:
%sql
-- ============================================================
-- PASO 3: Crear the Volume para archivos
-- ============================================================
-- Crear volume tipo MANAGED (Databricks administra el storage)
CREATE VOLUME IF NOT EXISTS bootcamp.landing.archivos
COMMENT 'Volume para almacenar archivos CSV crudos';
-- Verificar que se creó
SHOW VOLUMES IN bootcamp.landing;

In [0]:
%sql
-- ============================================================
-- PASO 4: Verificar que el archivo se subió correctamente
-- ============================================================
-- Listar archivos en el volume
LIST '/Volumes/bootcamp/landing/archivos/';

In [0]:
%sql
-- Leer directo del archivo es posible, y es útil también para analizarlo
SELECT * FROM read_files(
 '/Volumes/bootcamp/landing/archivos/properties_raw.csv',
 format => 'csv',
 header => true
 )

In [0]:
%sql
-- ============================================================
-- PASO 5: Crear la tabla properties_bronze
-- ============================================================
-- Primero, eliminamos la tabla si existe (para poder recrearla)
DROP TABLE IF EXISTS bootcamp.bronze.properties_bronze;
-- Crear tabla EXTERNA leyendo el CSV y nos quedamos solo con los registros que tienen url válida
CREATE TABLE bootcamp.bronze.properties_bronze
 SELECT * FROM read_files(
 '/Volumes/bootcamp/landing/archivos/properties_raw.csv',
 format => 'csv',
 header => true
 )
 where url like 'https%' --Filtramos por aquellos links que sean consistentes
 ;

In [0]:
%sql
-- Verificar que la tabla se creó correctamente
SHOW TABLES IN bootcamp.bronze;

# **PARTE 1: Exploración Inicial**

In [0]:
-- Cuántos registros hay en total en la tabla bootcamp.bronze.properties_bronze?
SELECT 
COUNT(*)
FROM 
bootcamp.bronze.properties_bronze;

In [0]:
-- ¿Qué columnas tiene la tabla? ¿Qué tipos de datos tienen?
DESCRIBE bootcamp.bronze.properties_bronze;

In [0]:
-- Ejercicio 1.3: Ver muestra de datos - Muestra las primeras 10 filas del dataset, pero solo las columnas más importantes: id, ubicacion, precio, expensas, tipo_de_operacion, moneda, ambientes, metros_cuadrados_totales, antiguedad, estado, zona. Pista: Usa SELECT con columnas específicas y LIMIT 10

SELECT ubicacion, precio, expensas, tipo_de_operacion, moneda, ambientes, metros_cuadrados_totales, antiguedad, estado, zona FROM bootcamp.bronze.properties_bronze LIMIT 10;

# PARTE 2: Análisis de Valores Nulos 

In [0]:
-- Ejercicio 2.1: Contar nulos por columna - ¿Cuántos valores nulos hay en cada columna importante? Calcula para: precio, expensas, tipo_de_operacion, moneda, ambientes, metros_cuadrados_totales, metros_cuadrados_cubiertos, orientacion_cardinal, piso, cochera, antiguedad, estado, zona. 

WITH total AS (
    SELECT COUNT(*) as n 
    FROM bootcamp.bronze.properties_bronze
),
nulos AS (
    SELECT
        COUNT(*) - COUNT(id) as nulos_id,
        COUNT(*) - COUNT(ubicacion) as nulos_ubicacion,
        COUNT(*) - COUNT(precio) as nulos_precio,
        COUNT(*) - COUNT(expensas) as nulos_expensas,
        COUNT(*) - COUNT(tipo_de_operacion) as nulos_tipo_operacion,
        COUNT(*) - COUNT(moneda) as nulos_moneda,
        COUNT(*) - COUNT(ambientes) as nulos_ambientes,
        COUNT(*) - COUNT(metros_cuadrados_totales) as nulos_m2_totales,
        COUNT(*) - COUNT(metros_cuadrados_cubiertos) as nulos_m2_cubiertos,
        COUNT(*) - COUNT(orientacion_cardinal) as nulos_orientacion_cardinal,
        COUNT(*) - COUNT(orientacion_inmueble) as nulos_orientacion_inmueble,
        COUNT(*) - COUNT(piso) as nulos_piso,
        COUNT(*) - COUNT(cochera) as nulos_cochera,
        COUNT(*) - COUNT(antiguedad) as nulos_antiguedad,
        COUNT(*) - COUNT(estado) as nulos_estado,
        COUNT(*) - COUNT(zona) as nulos_zona
    FROM bootcamp.bronze.properties_bronze
)
SELECT 
    t.n as total_registros,
    n.*
FROM total t, nulos n;

In [0]:
-- Ejercicio 2.2: Porcentaje de nulos - ¿Qué porcentaje de registros tienen valores nulos en las columnas clave?
WITH totales AS (
    SELECT COUNT(*) as total FROM bootcamp.bronze.properties_bronze
)
SELECT 
    ROUND((t.total - COUNT(precio)) * 100.0 / t.total, 2) as pct_nulos_precio,
    ROUND((t.total - COUNT(expensas)) * 100.0 / t.total, 2) as pct_nulos_expensas,
    ROUND((t.total - COUNT(ambientes)) * 100.0 / t.total, 2) as pct_nulos_ambientes,
    ROUND((t.total - COUNT(metros_cuadrados_totales)) * 100.0 / t.total, 2) as pct_nulos_m2_totales,
    ROUND((t.total - COUNT(metros_cuadrados_cubiertos)) * 100.0 / t.total, 2) as pct_nulos_m2_cubiertos,
    ROUND((t.total - COUNT(orientacion_cardinal)) * 100.0 / t.total, 2) as pct_nulos_orientacion,
    ROUND((t.total - COUNT(antiguedad)) * 100.0 / t.total, 2) as pct_nulos_antiguedad
FROM bootcamp.bronze.properties_bronze
CROSS JOIN totales t
GROUP BY t.total;

In [0]:
-- Ejercicio 2.3: Identificar columnas críticas - ¿Qué columnas tienen más del 50% de valores nulos? ¿Cuáles son críticas para el análisis? Pista: Usa el resultado del ejercicio anterior y filtra con HAVING o WHERE
WITH totales AS (
    SELECT 
        COUNT(*) AS total 
    FROM bootcamp.bronze.properties_bronze
),
pct_nulos AS (
SELECT 
        ROUND((t.total - COUNT(precio)) * 100.0 / t.total, 2) as pct_nulos_precio,
        ROUND((t.total - COUNT(expensas)) * 100.0 / t.total, 2) as pct_nulos_expensas,
        ROUND((t.total - COUNT(ambientes)) * 100.0 / t.total, 2) as pct_nulos_ambientes,
        ROUND((t.total - COUNT(metros_cuadrados_totales)) * 100.0 / t.total, 2) as pct_nulos_m2_totales,
        ROUND((t.total - COUNT(metros_cuadrados_cubiertos)) * 100.0 / t.total, 2) as pct_nulos_m2_cubiertos,
        ROUND((t.total - COUNT(orientacion_cardinal)) * 100.0 / t.total, 2) as pct_nulos_orientacion,
        ROUND((t.total - COUNT(antiguedad)) * 100.0 / t.total, 2) as pct_nulos_antiguedad,
        ROUND((t.total - COUNT(piso)) * 100.0 / t.total, 2) as pct_nulos_piso,
        ROUND((t.total - COUNT(cochera)) * 100.0 / t.total, 2) as pct_nulos_cochera
FROM bootcamp.bronze.properties_bronze
CROSS JOIN totales t
GROUP BY t.total
)

SELECT columna, porcentaje_nulos
FROM pct_nulos
UNPIVOT (
    porcentaje_nulos FOR columna IN (
        pct_nulos_precio, pct_nulos_expensas, pct_nulos_ambientes, 
        pct_nulos_m2_totales, pct_nulos_m2_cubiertos,
        pct_nulos_orientacion, pct_nulos_antiguedad, pct_nulos_piso, pct_nulos_cochera
    )
)
WHERE porcentaje_nulos > 50
ORDER BY porcentaje_nulos DESC;

# PARTE 3: Cardinalidad y Distribución


In [0]:
-- Ejercicio 3.1: Distribución de tipo de operación - ¿Qué valores únicos hay en tipo_de_operacion? ¿Cuántos registros hay de cada tipo? ¿Qué porcentaje representa cada uno? Pista: Usa GROUP BY, COUNT(*), y SUM(COUNT(*)) OVER() para calcular porcentajes

SELECT 
    tipo_de_operacion,
    COUNT(*) as n, 
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER()x, 2) as pct
FROM bootcamp.bronze.properties_bronze
GROUP BY tipo_de_operacion
ORDER BY n DESC;



In [0]:
-- Ejercicio 3.2: Distribución de moneda - ¿Qué monedas hay en el dataset? ¿Cuántos registros hay de cada una? ¿Hay alguna moneda que no esperabas? Pista: GROUP BY moneda con conteo y porcentaje

SELECT 
    moneda,
    COUNT(*) as n, 
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as pct
FROM bootcamp.bronze.properties_bronze
GROUP BY moneda
ORDER BY n;

In [0]:
-- Ejercicio 3.3: Distribución de ambientes - ¿Cuántas propiedades hay por cantidad de ambientes? ¿Cuál es la distribución? Ordena por cantidad de ambientes. Pista: GROUP BY ambientes ORDER BY ambientes

SELECT 
    ambientes, 
    COUNT(*) AS n,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct
FROM bootcamp.bronze.properties_bronze
GROUP BY ambientes
ORDER BY ambientes;
--

In [0]:
-- Ejercicio 3.4: Top zonas - ¿Cuáles son las 15 zonas con más propiedades? Muestra zona, cantidad y porcentaje del total. Pista: GROUP BY zona, COUNT(*), porcentaje, ORDER BY cantidad DESC LIMIT 15

SELECT 
    zona,
    COUNT(*) AS Cantidad_Propiedades,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct
FROM bootcamp.bronze.properties_bronze
GROUP BY zona
ORDER BY Cantidad_Propiedades DESC
LIMIT 15;

In [0]:
-- Ejercicio 3.5: Distribución de estado - ¿Qué valores hay en la columna estado? ¿Cuántas propiedades hay de cada estado? Pista: GROUP BY estado ORDER BY cantidad DESC
SELECT 
    estado,
    COUNT(*) AS Cantidad_Propiedades,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct
FROM bootcamp.bronze.properties_bronze
GROUP BY estado
ORDER BY Cantidad_Propiedades DESC;

# PARTE 4: Estadísticas Descriptivas de Variables Numéricas


In [0]:
-- Ejercicio 4.1: Estadísticas de precio por moneda y tipo de operación
-- ============================================================
-- EDA 4.2 (ANTES de limpiar): Estadísticas de METROS CUADRADOS (datos crudos)
-- ============================================================

SELECT 
    'metros_cuadrados_totales' as columna,
    COUNT(*) as cantidad_no_nulos,
    ROUND(MIN(metros_cuadrados_totales), 2) as minimo,
    ROUND(MAX(metros_cuadrados_totales), 2) as maximo,
    ROUND(AVG(metros_cuadrados_totales), 2) as promedio,
    ROUND(PERCENTILE(metros_cuadrados_totales, 0.5), 2) as mediana
FROM bootcamp.bronze.properties_bronze
WHERE metros_cuadrados_totales IS NOT NULL AND metros_cuadrados_totales > 0

UNION ALL

SELECT 
    'metros_cuadrados_cubiertos' as columna,
    COUNT(*) as cantidad_no_nulos,
    ROUND(MIN(metros_cuadrados_cubiertos), 2) as minimo,
    ROUND(MAX(metros_cuadrados_cubiertos), 2) as maximo,
    ROUND(AVG(metros_cuadrados_cubiertos), 2) as promedio,
    ROUND(PERCENTILE(metros_cuadrados_cubiertos, 0.5), 2) as mediana
FROM bootcamp.bronze.properties_bronze
WHERE metros_cuadrados_cubiertos IS NOT NULL AND metros_cuadrados_cubiertos > 0;

In [0]:
%sql
create or replace temporary View propiedades_clean as
select 
CASE
  WHEN precio RLIKE '^[^a-zA-Z]+$' THEN precio::double
  ELSE NULL
  END as precio,
  moneda,
CASE
  WHEN ambientes RLIKE '^[^a-zA-Z]+$' THEN ambientes::double
  ELSE NULL
END as ambientes
 ,
 CASE
  WHEN metros_cuadrados_totales RLIKE '^[^a-zA-Z]+$' THEN metros_cuadrados_totales::double
  ELSE NULL
END as metros_cuadrados_totales
 ,
 CASE
  WHEN metros_cuadrados_cubiertos RLIKE '^[^a-zA-Z]+$' THEN metros_cuadrados_cubiertos::double
  ELSE NULL
END as metros_cuadrados_cubiertos
 ,
 CASE
  WHEN antiguedad RLIKE '^[^a-zA-Z]+$' THEN antiguedad::double
  ELSE NULL
END as antiguedad,
tipo_de_operacion
,id
,ubicacion
,numero
,calle
,expensas
,orientacion_cardinal
,orientacion_inmueble
,piso
,cochera
,estado
,tipo_vendedor
,url
,zona
,fecha
,hora
from bootcamp.bronze.properties_bronze

In [0]:
-- ============================================================
-- EDA 4.1: Estadísticas de PRECIO (separado por moneda y tipo de operación)
-- ============================================================

SELECT 
    moneda,
    tipo_de_operacion,
    COUNT(*) as cantidad,
    ROUND(MIN(precio), 2) as precio_min,
    ROUND(MAX(precio), 2) as precio_max,
    ROUND(AVG(precio), 2) as precio_promedio,
    ROUND(PERCENTILE(precio, 0.5), 2) as precio_mediana,
    ROUND(PERCENTILE(precio, 0.25), 2) as precio_p25,
    ROUND(PERCENTILE(precio, 0.75), 2) as precio_p75
FROM propiedades_clean
WHERE precio IS NOT NULL AND precio > 0
GROUP BY moneda, tipo_de_operacion
ORDER BY cantidad DESC;

In [0]:
%sql
-- ============================================================
-- EDA 4.1b (BONUS — no está en el PDF): Estadísticas de PRECIO (filtrado)
-- Solo USD/ARS, operaciones principales, sin outliers (P1-P99)
-- ============================================================
    WITH limites AS (
    SELECT 
      moneda,
      tipo_de_operacion,
      PERCENTILE(precio, 0.01) AS p01,
      PERCENTILE(precio, 0.99) AS p99
    FROM propiedades_clean
    WHERE precio > 0
      AND moneda IN ('USD', 'ARS')
      AND tipo_de_operacion IN ('venta', 'alquiler')
    GROUP BY moneda, tipo_de_operacion
  )
  SELECT 
    p.moneda,
    p.tipo_de_operacion,
    COUNT(*) AS cantidad,
    ROUND(MIN(p.precio), 2) AS precio_min,
    ROUND(MAX(p.precio), 2) AS precio_max,
    ROUND(AVG(p.precio), 2) AS precio_promedio,
    ROUND(PERCENTILE(p.precio, 0.5), 2) AS precio_mediana,
    ROUND(PERCENTILE(p.precio, 0.25), 2) AS precio_p25,
    ROUND(PERCENTILE(p.precio, 0.75), 2) AS precio_p75
  FROM propiedades_clean p
  JOIN limites l 
    ON p.moneda = l.moneda 
    AND p.tipo_de_operacion = l.tipo_de_operacion
  WHERE p.precio BETWEEN l.p01 AND l.p99
  GROUP BY p.moneda, p.tipo_de_operacion
  ORDER BY cantidad DESC;

In [0]:
%sql
-- ============================================================
-- Creamos la nueva vista temporal
-- ============================================================
CREATE OR REPLACE TEMPORARY VIEW propiedades_clean_2 AS
(
    WITH limites AS (
    SELECT 
      moneda,
      tipo_de_operacion,
      PERCENTILE(precio, 0.01) AS p01,
      PERCENTILE(precio, 0.99) AS p99
    FROM propiedades_clean
    WHERE precio > 0
      AND moneda IN ('USD', 'ARS')
      AND tipo_de_operacion IN ('venta', 'alquiler')
    GROUP BY moneda, tipo_de_operacion
  )
  SELECT p.*
  FROM propiedades_clean p
  JOIN limites l 
    ON p.moneda = l.moneda 
    AND p.tipo_de_operacion = l.tipo_de_operacion
  WHERE p.precio BETWEEN l.p01 AND l.p99
);

In [0]:
-- Ejercicio 4.2: Estadísticas de metros cuadrados
SELECT 
    'metros_cuadrados_totales' as columna,
    COUNT(*) as cantidad,
    ROUND(MIN(metros_cuadrados_totales), 2) as metros_min,
    ROUND(MAX(metros_cuadrados_totales), 2) as metros_max,
    ROUND(AVG(metros_cuadrados_totales), 2) as metros_promedio,
    ROUND(PERCENTILE(metros_cuadrados_totales, 0.5), 2) as metros_mediana,
    ROUND(PERCENTILE(metros_cuadrados_totales, 0.25), 2) as metros_p25,
    ROUND(PERCENTILE(metros_cuadrados_totales, 0.75), 2) as metros_p75
FROM propiedades_clean_2
WHERE metros_cuadrados_totales IS NOT NULL AND metros_cuadrados_totales > 0

UNION ALL

SELECT 
    'metros_cuadrados_cubiertos' as columna,
    COUNT(*) as cantidad_no_nulos,
    ROUND(MIN(metros_cuadrados_cubiertos), 2) as minimo,
    ROUND(MAX(metros_cuadrados_cubiertos), 2) as maximo,
    ROUND(AVG(metros_cuadrados_cubiertos), 2) as promedio,
    ROUND(PERCENTILE(metros_cuadrados_cubiertos, 0.5), 2) as mediana,
    ROUND(PERCENTILE(metros_cuadrados_cubiertos, 0.25), 2) as metros_p25,
    ROUND(PERCENTILE(metros_cuadrados_cubiertos, 0.75), 2) as metros_p75
FROM propiedades_clean_2
WHERE metros_cuadrados_cubiertos IS NOT NULL AND metros_cuadrados_cubiertos > 0;

In [0]:
-- Ejercicio 4.3: Análisis de antigüedad
SELECT 
    antiguedad,
    COUNT (*) as cantidad
FROM propiedades_clean_2
WHERE antiguedad IS NOT NULL AND antiguedad > 0
GROUP BY antiguedad
ORDER BY antiguedad DESC;

In [0]:
-- Ejercicio 4.4: Estadísticas de antigüedad sin placeholders
SELECT 
    COUNT(*) as cantidad,
    MIN(CAST(antiguedad AS INT)) as antiguedad_min,
    MAX(CAST(antiguedad AS INT)) as antiguedad_max,
    ROUND(AVG(CAST(antiguedad AS INT)), 2) as antiguedad_promedio,
    ROUND(PERCENTILE(CAST(antiguedad AS INT), 0.5), 0) as antiguedad_mediana
FROM propiedades_clean_2
WHERE antiguedad IS NOT NULL 
  AND antiguedad != '999' 
  AND antiguedad >= 0;

# PARTE 5: Detección de Problemas de Calidad

In [0]:
-- Ejercicio 5.1
-- Reporte de calidad usando CTEs
-- Crea un reporte que muestre: Total de registros, Cantidad de precios inválidos (NULL o <= 0), Cantidad de metros cuadrados inválidos (NULL o <= 0), Cantidad de antigüedad con valor 999 (placeholder), Cantidad de ambientes inválidos (NULL o 0), Cantidad de moneda vacía, Cantidad de tipo_operacion vacía, Porcentaje de cada problema sobre el total. Pista: Usa CTEs para organizar. Calcula totales y problemas por separado, luego únelos.
